# Step 4 — QAOA on the pruning Ising Hamiltonian

This notebook is **Step 4** of the quantum-pruning pipeline.

- **Step 2** (`qnn_and_pruning.ipynb`) measured per-block pruning sensitivity → `cost_loss_table.csv`.
- **Step 3** (`qubo_hameltonian.ipynb`) turned that into a QUBO and an **Ising Hamiltonian** plus a classical brute-force optimum.
- **Step 4 (this file)** runs **QAOA** (Qiskit) on that Hamiltonian and checks that the quantum result matches the known classical optimum.

Binary variable convention (same as Step 3): `x_i = 1` → **prune** block *i*, `x_i = 0` → **keep** it, with `x_i = (1 - Z_i) / 2`.

> **Note:** With only 8 qubits this is a *simulation / demonstration*. The whole search space (2^8 = 256 states) is tiny, so the classical brute force in `qubo_energy_check.csv` is the exact global optimum. QAOA should reproduce it — that is our PASS/FAIL test.

In [ ]:
# --- Assumed package versions --------------------------------------------
#   qiskit            >= 2.0   (uses qiskit.primitives.StatevectorSampler,
#                               a BaseSamplerV2 reference primitive, + SparsePauliOp)
#   qiskit-algorithms >= 0.4   (QAOA now takes a BaseSamplerV2 sampler;
#                               also COBYLA, algorithm_globals)
#
# NOTE: qiskit 2.x REMOVED the old V1 `qiskit.primitives.Sampler`. We therefore
# use `StatevectorSampler` (V2). The reference StatevectorSampler is exact and
# needs no Aer backend, so qiskit-aer is NOT required for this notebook.
#
# If anything is missing, uncomment and run:
# %pip install "qiskit>=2.0" "qiskit-algorithms>=0.4"
import qiskit
import qiskit_algorithms

print("qiskit           :", qiskit.__version__)
print("qiskit-algorithms:", qiskit_algorithms.__version__)

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np

from qiskit.primitives import StatevectorSampler   # V2 reference sampler (qiskit >= 2.0)
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals

In [ ]:
# --- Configuration (argparse, but safe inside Jupyter) -------------------
# parse_known_args() ignores the kernel's own --f=...kernel.json argument.
parser = argparse.ArgumentParser(description="Step 4: QAOA on the pruning Ising Hamiltonian.")
parser.add_argument("--indir", type=str, default="qubo_outputs",
                    help="Directory holding the Step-3 outputs.")
parser.add_argument("--reps", type=int, default=2,
                    help="Number of QAOA layers (p). Deeper = more expressive, slower.")
parser.add_argument("--shots", type=int, default=1024,
                    help="Sampler shots per circuit evaluation.")
parser.add_argument("--seed", type=int, default=42,
                    help="Random seed for reproducibility.")
parser.add_argument("--maxiter", type=int, default=200,
                    help="Max iterations for the COBYLA classical optimizer.")

args, _unknown = parser.parse_known_args()

INDIR = Path(args.indir)
print("Config:", vars(args))

# Make the whole run reproducible.
algorithm_globals.random_seed = args.seed

In [ ]:
# --- Robust file loading -------------------------------------------------
def _require(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}\n"
            f"Run Step 3 (qubo_hameltonian.ipynb) first to generate it."
        )
    return path


def load_hamiltonian(indir: Path) -> Dict[str, Any]:
    """Load the Ising terms produced in Step 3."""
    path = _require(indir / "hamiltonian_terms.json")
    with path.open("r", encoding="utf-8") as f:
        ham = json.load(f)
    # Infer number of qubits from the highest index that appears anywhere.
    max_i = 0
    for t in ham.get("z_terms", []):
        max_i = max(max_i, t["i"])
    for t in ham.get("zz_terms", []):
        max_i = max(max_i, t["i"], t["j"])
    ham["n_qubits"] = max_i + 1
    return ham


def load_candidates(indir: Path) -> List[Dict[str, Any]]:
    """Load the selected candidates, ordered by qubit_index (x_0 first)."""
    path = _require(indir / "selected_candidates.csv")
    rows: List[Dict[str, Any]] = []
    with path.open("r", newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows.append({
                "qubit_index": int(r["qubit_index"]),
                "candidate": r["candidate"],
                "params": int(float(r["params"])),
                "loss_penalty": float(r["loss_penalty"]),
                "compression_value": float(r["compression_value"]),
            })
    rows.sort(key=lambda r: r["qubit_index"])
    return rows


def load_brute_force_optimum(indir: Path) -> Dict[str, Any]:
    """Load the top (lowest-energy) row of the classical brute-force check."""
    path = _require(indir / "qubo_energy_check.csv")
    with path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        best = next(reader)  # file is already sorted ascending by energy
    return {
        # NOTE: this bitstring is in candidate order, i.e. x_0 is the LEFT-most char.
        "bitstring": best["bitstring"].strip(),
        "energy": float(best["energy"]),
        "pruned_blocks": best["pruned_blocks"],
    }


ham = load_hamiltonian(INDIR)
candidates = load_candidates(INDIR)
bf = load_brute_force_optimum(INDIR)

N = ham["n_qubits"]
assert N == len(candidates), f"Hamiltonian has {N} qubits but {len(candidates)} candidates."

print(f"Loaded Hamiltonian on {N} qubits.")
print(f"Classical optimum  : {bf['bitstring']}  energy={bf['energy']:.6f}  -> {bf['pruned_blocks']}")

In [ ]:
# --- Build the SparsePauliOp from the Ising terms ------------------------
# Ising form:  H = constant + sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
#
# We build the operator WITHOUT the identity (constant) term: a constant only
# shifts every energy equally and does not change the optimizer's argmin.
# We add the constant back whenever we REPORT an energy.
#
# Endianness: SparsePauliOp.from_sparse_list places a Pauli on the exact qubit
# index we give, so indices line up 1:1 with `qubit_index` in the candidates.
CONST = float(ham["constant"])


def build_operator(ham: Dict[str, Any]) -> SparsePauliOp:
    n = ham["n_qubits"]
    sparse_list = []
    for t in ham["z_terms"]:
        sparse_list.append(("Z", [t["i"]], float(t["coefficient"])))
    for t in ham["zz_terms"]:
        sparse_list.append(("ZZ", [t["i"], t["j"]], float(t["coefficient"])))
    return SparsePauliOp.from_sparse_list(sparse_list, num_qubits=n)


def ising_energy(x: List[int]) -> float:
    """Exact energy of a binary assignment x (x_i in {0,1}), including the constant.

    Uses Z_i = 1 - 2 x_i, so x=0 -> Z=+1 and x=1 -> Z=-1. This reproduces the
    QUBO energy column in qubo_energy_check.csv.
    """
    z = [1 - 2 * int(b) for b in x]
    e = CONST
    for t in ham["z_terms"]:
        e += t["coefficient"] * z[t["i"]]
    for t in ham["zz_terms"]:
        e += t["coefficient"] * z[t["i"]] * z[t["j"]]
    return float(e)


operator = build_operator(ham)
print(f"Operator: {len(operator)} Pauli terms, {operator.num_qubits} qubits (constant {CONST:.6f} held separately).")

# Self-check: our ising_energy must reproduce the brute-force optimum energy.
_bf_x = [int(c) for c in bf["bitstring"]]            # candidate order: x_0 first
_check = ising_energy(_bf_x)
print(f"Sanity check on classical optimum: recomputed energy={_check:.6f} (file {bf['energy']:.6f})")
assert abs(_check - bf["energy"]) < 1e-6, "Energy function does not match Step-3 output!"

In [ ]:
# --- Run QAOA ------------------------------------------------------------
# StatevectorSampler is the V2 reference primitive: shots are configured via
# `default_shots`, and `seed` makes the sampling reproducible.
sampler = StatevectorSampler(default_shots=args.shots, seed=args.seed)
optimizer = COBYLA(maxiter=args.maxiter)

# A callback lets us record the optimizer's progress so we can plot convergence.
# Signature is fixed by qiskit_algorithms: (eval_count, params, mean, metadata).
# `mean` is the expected energy of the operator (WITHOUT our separate constant).
history: Dict[str, list] = {"evals": [], "energies": []}
def _callback(eval_count, params, mean, metadata):
    history["evals"].append(eval_count)
    history["energies"].append(float(mean))

qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=args.reps, callback=_callback)

print(f"Running QAOA (reps={args.reps}, shots={args.shots}, maxiter={args.maxiter}) ...")
result = qaoa.compute_minimum_eigenvalue(operator)
print("QAOA finished.")
print("Optimal circuit parameters:", np.round(result.optimal_point, 4).tolist())

# `optimal_value` is the EXPECTED energy of the final circuit (without our
# separately-held constant). The single best sample QAOA actually observed is in
# `result.best_measurement` -- that is the standard, robust way to report QAOA's
# answer (far less noisy than the most-probable bitstring on shallow circuits).
print("Expectation value (no constant):", round(float(result.optimal_value), 6))

In [ ]:
# --- Decode the result ---------------------------------------------------
# Qiskit measurement bitstrings are LITTLE-ENDIAN: the right-most char is qubit 0.
# Our candidate order (and the brute-force CSV) is BIG-ENDIAN: x_0 is left-most.
# We verified this convention empirically (X on qubit 0 -> counts '...1').
def little_endian_to_x(bitstring_le: str, n: int) -> List[int]:
    """Convert a little-endian qiskit bitstring to x indexed by qubit (x_0 first)."""
    bs = bitstring_le.zfill(n)
    return [int(bs[n - 1 - i]) for i in range(n)]


def probability_distribution(result, n: int) -> Dict[str, float]:
    """Return {little_endian_bitstring: probability} robustly across qiskit versions."""
    es = result.eigenstate
    if hasattr(es, "binary_probabilities"):
        return es.binary_probabilities()
    out: Dict[str, float] = {}
    for k, v in dict(es).items():
        bs = format(int(k), f"0{n}b") if isinstance(k, (int, np.integer)) else str(k).zfill(n)
        out[bs] = out.get(bs, 0.0) + float(v)
    return out


# PRIMARY answer: the lowest-energy sample QAOA actually observed.
# `best_measurement["value"]` excludes our constant, so we recompute the full
# energy with ising_energy() to stay consistent with qubo_energy_check.csv.
best = result.best_measurement
best_le = str(best["bitstring"]).zfill(N)               # little-endian (qubit 0 right-most)
x = little_endian_to_x(best_le, N)                      # qubit order, x_0 first
x_string = "".join(str(b) for b in x)                   # matches brute-force CSV format
qaoa_energy = ising_energy(x)

# Probability mass that QAOA put on this winning bitstring.
dist = probability_distribution(result, N)
mp_prob = float(dist.get(best_le, float(best.get("probability", 0.0))))

# Also report the most-probable bitstring for reference (can differ from best on
# shallow/low-shot runs; best_measurement is the one we trust).
most_probable_le = max(dist, key=dist.get)

pruned = [c["candidate"] for c in candidates if x[c["qubit_index"]] == 1]
total_compression = float(sum(c["compression_value"] for c in candidates if x[c["qubit_index"]] == 1))
total_loss = float(sum(c["loss_penalty"] for c in candidates if x[c["qubit_index"]] == 1))

print(f"Best sampled bitstring (x_0 first): {x_string}   p={mp_prob:.4f}")
print(f"Most-probable bitstring (ref)     : "
      f"{''.join(str(b) for b in little_endian_to_x(most_probable_le, N))}")
print(f"QAOA energy        : {qaoa_energy:.6f}")
print(f"Pruned blocks      : {pruned}")

In [ ]:
# --- Validation against the classical optimum ----------------------------
energy_gap = qaoa_energy - bf["energy"]
found_optimum = abs(energy_gap) < 1e-6 and x_string == bf["bitstring"]

print("=" * 60)
print("VALIDATION")
print("=" * 60)
print(f"QAOA solution      : {x_string}  energy={qaoa_energy:.6f}")
print(f"Classical optimum  : {bf['bitstring']}  energy={bf['energy']:.6f}")
print(f"Energy gap         : {energy_gap:+.6e}")
print(f"Result             : {'PASS - QAOA found the global optimum' if found_optimum else 'FAIL - QAOA did NOT match the optimum'}")
print("-" * 60)
print(f"Total compression  : {total_compression:.4f}  (sum of C_i over pruned blocks)")
print(f"Total loss penalty : {total_loss:.6f}  (sum of L_i over pruned blocks)")

if not found_optimum:
    print("\nTip: QAOA is heuristic. Try increasing --reps (e.g. 3-5) or --maxiter,")
    print("or re-run with a different --seed. On 8 qubits the optimum is easily reachable.")

In [ ]:
# --- Save results --------------------------------------------------------
out_path = INDIR / "qaoa_result.json"
payload = {
    "best_bitstring": x_string,                 # x_0 first (matches selected_candidates.csv)
    "best_bitstring_little_endian": best_le,    # qiskit order (qubit 0 right-most)
    "energy": qaoa_energy,
    "probability": mp_prob,
    "pruned_blocks": pruned,
    "num_pruned_blocks": int(sum(x)),
    "total_compression": total_compression,
    "total_loss_penalty": total_loss,
    "validation": {
        "classical_optimum_bitstring": bf["bitstring"],
        "classical_optimum_energy": bf["energy"],
        "energy_gap": energy_gap,
        "found_global_optimum": bool(found_optimum),
    },
    "optimizer": {
        "method": "COBYLA",
        "reps": args.reps,
        "shots": args.shots,
        "seed": args.seed,
        "maxiter": args.maxiter,
        "optimal_point": list(map(float, result.optimal_point)),
        "optimal_value_no_constant": float(result.optimal_value),
    },
}

with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(f"Saved QAOA result to: {out_path}")
print(json.dumps(payload, indent=2))

## Visualizing the QAOA

Three qiskit-native views of what just happened:

1. **The QAOA ansatz** — the parameterized quantum circuit (cost layer + mixer layer, repeated `reps` times) that QAOA optimized.
2. **Optimizer convergence** — how the expected energy fell as COBYLA tuned the circuit angles.
3. **Sampled bitstrings** — the probability QAOA assigned to each candidate solution, with the winning (lowest-energy) one highlighted.

In [ ]:
# --- 1) Draw the QAOA quantum circuit that was used ----------------------
# Same idiom as the basic Qiskit example:
#
#     from qiskit import QuantumCircuit
#     qc = QuantumCircuit(2)
#     qc.h(0)
#     qc.cx(0, 1)
#     qc.draw(output='mpl')
#
# Here the circuit is QAOA's ansatz: an H on every qubit (equal superposition),
# then `reps` repetitions of a cost layer (gamma * Z/ZZ from our Hamiltonian)
# followed by a mixer layer (RX(beta) on each qubit). `.decompose()` expands the
# high-level cost/mixer blocks down to basic gates so the structure is visible.
from qiskit import QuantumCircuit  # (same import as the reference snippet)

# The exact circuit QAOA used, with the optimal angles bound in:
qc = qaoa.ansatz.assign_parameters(result.optimal_point)

print(f"QAOA circuit: {qc.num_qubits} qubits, depth {qc.decompose().depth()}, "
      f"{qaoa.ansatz.num_parameters} tunable parameters (now bound).")

# Display the circuit (decomposed into basic gates), exactly like qc.draw(output='mpl').
qc.decompose().draw(output='mpl', fold=-1)

In [ ]:
# --- 2) Optimizer convergence --------------------------------------------
# `history` was filled by the callback during the QAOA run. We add CONST back so
# the y-axis is on the same energy scale as the brute-force optimum, and draw a
# dashed line at the known global minimum for reference.
energies_with_const = [e + CONST for e in history["energies"]]

plt.figure(figsize=(8, 4.5))
plt.plot(history["evals"], energies_with_const, marker="o", ms=3, lw=1, label="QAOA expected energy")
plt.axhline(bf["energy"], color="red", ls="--", lw=1.2,
            label=f"classical optimum ({bf['energy']:.4f})")
plt.xlabel("optimizer evaluation")
plt.ylabel("energy (incl. constant)")
plt.title("QAOA / COBYLA convergence")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3) Histogram of sampled bitstrings ----------------------------------
# qiskit's plot_histogram expects {bitstring: probability}. We relabel the keys
# from qiskit's little-endian order into our big-endian "x_0 first" convention so
# the bars match selected_candidates.csv and the brute-force CSV.
from qiskit.visualization import plot_histogram

big_endian_dist = {
    "".join(str(b) for b in little_endian_to_x(bs_le, N)): p
    for bs_le, p in dist.items()
}

# Keep the most probable handful so the chart stays readable.
top_items = sorted(big_endian_dist.items(), key=lambda kv: kv[1], reverse=True)[:12]
top = dict(top_items)

hist_fig = plot_histogram(
    top,
    figsize=(11, 5),
    title=f"QAOA sampled bitstrings (top {len(top)}) — winner {x_string} highlighted",
    sort="value_desc",
    bar_labels=False,
)
# Highlight the winning (lowest-energy) bitstring in red.
ax = hist_fig.axes[0]
for label, patch in zip([t.get_text() for t in ax.get_xticklabels()], ax.patches):
    if label == x_string:
        patch.set_color("crimson")
plt.show()

## Step 5 — Apply the QAOA decision to the real model

So far everything has been abstract optimization. Now we **act on the result**: take the
blocks QAOA chose to prune (`pruned`), remove them from the actual fine-tuned ConvNeXT,
and measure what really happens to accuracy / loss / F1 on the Canadian-streetview test set.

"Removing" a block means replacing it with an **identity bypass** (`forward(x) → x`) — the
same simulation used in Step 2. Because input and output shapes match inside a stage, this
approximates deleting the block while keeping the tensor path intact.

> **Why this matters:** Step 2 measured each block's damage *individually*, and the QUBO
> simply *summed* those penalties (no interaction term). This cell is the empirical check of
> that assumption: we compare the **predicted** drop (sum of individual sensitivities) against
> the **actual** drop when all chosen blocks are pruned **together**.
>
> Requires the ML stack: `pip install torch torchvision timm huggingface_hub datasets scikit-learn`.
> The model (~110 MB) and a test-set subset are downloaded from the Hugging Face Hub on first run.

In [ ]:
# --- ML helpers (same preprocessing / eval / bypass logic as Step 2) ------
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, f1_score
import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download


def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    """ConvNeXT preprocessing: thumbnail + center-pad onto a black canvas."""
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", target_size, (0, 0, 0))
    canvas.paste(img, ((target_size[0] - img.size[0]) // 2,
                       (target_size[1] - img.size[1]) // 2))
    return canvas


CONVNEXT_TRANSFORM = v2.Compose([
    v2.Lambda(lambda img: resize_and_pad(img)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class StreetViewSubset(Dataset):
    """Small in-memory subset of the HF dataset for a fast evaluation."""
    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        return self.transform(img.convert("RGB")), int(row["label"])


def load_convnext(device: torch.device) -> nn.Module:
    """Load the fine-tuned ConvNeXT-tiny checkpoint from the HF Hub."""
    path = hf_hub_download(
        repo_id="canada-guesser/canadian_streetview_cities_models",
        filename="cnn_model/convnext_tiny_set_3_final.bin",
    )
    model = timm.create_model("convnext_tiny", pretrained=False, num_classes=15)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    return model.to(device).eval()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    """Mean cross-entropy loss, accuracy and macro-F1 over the loader."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, total = 0.0, 0
    preds, labels = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += float(criterion(logits, yb).item())
        total += int(yb.numel())
        preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        labels.extend(yb.cpu().tolist())
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


class CandidateWrapper(nn.Module):
    """Wraps a block so it can be bypassed (forward(x) -> x) to simulate pruning."""
    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module
        self.bypass = False

    def forward(self, x, *args, **kwargs):
        return x if self.bypass else self.module(x, *args, **kwargs)


def get_module(model: nn.Module, name: str) -> nn.Module:
    """Resolve a dotted module path like 'stages.3.blocks.2'."""
    mod = model
    for part in name.split("."):
        mod = mod[int(part)] if part.isdigit() else getattr(mod, part)
    return mod


def replace_module(model: nn.Module, name: str, new_module: nn.Module) -> None:
    """Replace the submodule at a dotted path with new_module (in place)."""
    parts = name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = parent[int(part)] if part.isdigit() else getattr(parent, part)
    key = parts[-1]
    if key.isdigit():
        parent[int(key)] = new_module
    else:
        setattr(parent, key, new_module)

print("ML helpers ready.")

In [ ]:
# --- Load the model + test subset, then measure the baseline -------------
# MAX_SAMPLES trades speed vs. statistical stability. Step 2 used 600; raise this
# (or use the full split) for final numbers. On CPU each evaluation is a few minutes.
MAX_SAMPLES = 400
BATCH_SIZE = 16

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {torch_device}")

print("Loading fine-tuned ConvNeXT-tiny ...")
prune_model = load_convnext(torch_device)

print(f"Loading test[:{MAX_SAMPLES}] subset ...")
test_ds = load_dataset("canada-guesser/Canadian-streetview-cities", split=f"test[:{MAX_SAMPLES}]")
test_loader = DataLoader(
    StreetViewSubset(test_ds, CONVNEXT_TRANSFORM),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

print("Evaluating baseline (unpruned) model ...")
baseline_metrics = evaluate(prune_model, test_loader, torch_device)
print(f"Baseline: loss={baseline_metrics['loss']:.6f}  "
      f"acc={baseline_metrics['accuracy']:.4f}  "
      f"f1={baseline_metrics['macro_f1']:.4f}  n={baseline_metrics['n_samples']}")

In [ ]:
# --- Prune the QAOA-selected blocks together, then re-evaluate -----------
# `pruned` came from the QAOA decode cell. Wrap each chosen block and bypass it,
# so the forward pass skips all of them simultaneously.
prune_targets = list(pruned)
print(f"QAOA chose to prune {len(prune_targets)} block(s): {prune_targets}\n")

wrappers = {}
for name in prune_targets:
    wrapper = CandidateWrapper(get_module(prune_model, name))
    replace_module(prune_model, name, wrapper)
    wrapper.bypass = True              # bypass = simulate removal
    wrappers[name] = wrapper

print("Evaluating pruned model ...")
pruned_metrics = evaluate(prune_model, test_loader, torch_device)

# Restore the model (un-bypass) so the cell is safe to re-run.
for name, wrapper in wrappers.items():
    wrapper.bypass = False

# Actual, measured impact of removing all chosen blocks at once.
actual_loss_inc = pruned_metrics["loss"] - baseline_metrics["loss"]
actual_acc_drop = baseline_metrics["accuracy"] - pruned_metrics["accuracy"]
actual_f1_drop = baseline_metrics["macro_f1"] - pruned_metrics["macro_f1"]

# Predicted impact = sum of the INDIVIDUAL per-block sensitivities from Step 2.
# (This is exactly what the QUBO assumed: no interaction between blocks.)
predicted_loss_inc, predicted_acc_drop = 0.0, 0.0
sens_path = Path("cost_loss_table.csv")
if sens_path.exists():
    with sens_path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("candidate") in prune_targets:
                predicted_loss_inc += float(row.get("loss_increase_raw", 0.0) or 0.0)
                predicted_acc_drop += float(row.get("accuracy_drop_raw", 0.0) or 0.0)

params_saved = sum(
    int(c["params"]) for c in candidates if c["candidate"] in prune_targets
)

print("\n" + "=" * 64)
print("REAL-MODEL PRUNING RESULT")
print("=" * 64)
print(f"{'metric':<16}{'baseline':>12}{'pruned':>12}{'change':>12}")
print("-" * 64)
print(f"{'loss':<16}{baseline_metrics['loss']:>12.4f}{pruned_metrics['loss']:>12.4f}{actual_loss_inc:>+12.4f}")
print(f"{'accuracy':<16}{baseline_metrics['accuracy']:>12.4f}{pruned_metrics['accuracy']:>12.4f}{-actual_acc_drop:>+12.4f}")
print(f"{'macro_f1':<16}{baseline_metrics['macro_f1']:>12.4f}{pruned_metrics['macro_f1']:>12.4f}{-actual_f1_drop:>+12.4f}")
print("-" * 64)
print(f"Parameters removed : {params_saved:,}")
print(f"\nPredicted vs actual (tests the QUBO's no-interaction assumption):")
print(f"  loss increase : predicted(sum of singles)={predicted_loss_inc:.4f}   actual(together)={actual_loss_inc:.4f}")
print(f"  accuracy drop : predicted(sum of singles)={predicted_acc_drop:.4f}   actual(together)={actual_acc_drop:.4f}")
if actual_loss_inc > predicted_loss_inc * 1.5 and predicted_loss_inc > 0:
    print("  -> Actual damage notably exceeds the sum: blocks interact. Consider")
    print("     adding pairwise interaction terms to the QUBO for higher fidelity.")
else:
    print("  -> Actual damage is close to the sum: the linear QUBO assumption holds well.")

# Persist alongside the QAOA result.
with (INDIR / "pruning_eval.json").open("w", encoding="utf-8") as f:
    json.dump({
        "pruned_blocks": prune_targets,
        "params_removed": params_saved,
        "max_samples": MAX_SAMPLES,
        "baseline": baseline_metrics,
        "pruned": pruned_metrics,
        "actual": {"loss_increase": actual_loss_inc,
                   "accuracy_drop": actual_acc_drop,
                   "f1_drop": actual_f1_drop},
        "predicted_from_step2": {"loss_increase": predicted_loss_inc,
                                 "accuracy_drop": predicted_acc_drop},
    }, f, indent=2)
print(f"\nSaved real-model evaluation to: {INDIR / 'pruning_eval.json'}")